# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walk-through for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Display dataset title and description
print(f"Dataset title: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will inspect the record sets provided in the dataset. Each record set, field, and column is referred to by its `@id`.

In [ ]:
# List available record sets by their @id and name

record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"  @id: {rs.id} | Name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll load the main experimental record set that contains proteomics data (`@id` may vary depending on the dataset definition; commonly, it is something like `dv:ExperimentDataset`).

In [ ]:
# Identify main record set to load - replace with actual @id from the listing above as needed
# (for this dataset, typically: 'dv:ExperimentDataset')
experiment_record_set_id = None
for rs in dataset.record_sets:
    if rs.id == 'dv:ExperimentDataset':
        experiment_record_set_id = rs.id
        break

if experiment_record_set_id is None:
    raise ValueError('Main experiment record set not found.')

# Load all records into a DataFrame
records = list(dataset.records(record_set=experiment_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from record set @{experiment_record_set_id}.")
print("Columns:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field usually present in such proteomics quantification datasets (e.g., 'cr:Abundance' or 'cr:MW'), filter, normalize, and group data. 

Adjust the `numeric_field_id` and `group_field_id` values as per previously printed DataFrame columns.

In [ ]:
# Assign an existing numeric field @id as per previous overview (e.g., 'cr:MW', 'cr:Abundance', 'cr:PeptideCount') 
# and a grouping field (e.g., 'cr:Description' or 'cr:Gene')

numeric_field_id = None
possible_numeric_fields = ['cr:MW', 'cr:Abundance', 'cr:PeptideCount', 'cr:Coverage']
for f in possible_numeric_fields:
    if f in df.columns:
        numeric_field_id = f
        break
if numeric_field_id is None:
    raise ValueError('No suitable numeric field found in record set.')

# Group by a categorical field if available
group_field_id = None
possible_group_fields = ['cr:Gene', 'cr:Description', 'cr:Protein']
for f in possible_group_fields:
    if f in df.columns:
        group_field_id = f
        break

print(f"Using numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Grouping by: {group_field_id}")

# Filtering
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")

# Normalization
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(f"Grouped data by {group_field_id}, showing means:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll create a histogram for the chosen numeric field and a boxplot grouped by the selected field (where possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot grouped by group_field_id
if group_field_id and group_field_id in df.columns:
    top_values = df[group_field_id].value_counts().head(8).index
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df[df[group_field_id].isin(top_values)],
                x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id} (top 8 groups)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We explored the FAIR^2 dataset on mass spectrometry of extracellular vesicles from human mast cells. Using `mlcroissant`, we loaded the data by referencing all entities using their `@id`. We presented record set structure, extracted data, and performed basic filtering, normalization, grouping, and visualization. This pipeline allows for easy extension and deeper biological or analytical insights into protein expression and related attributes in the dataset.